## Import et lecture

In [1]:
import pandas as pd
from pathlib import Path

# Chemin vers le CSV (depuis notebooks/, on remonte d'un cran)
CSV_PATH = Path("..") / "data" / "healthcare_dataset.csv"

assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH.resolve()}"

df = pd.read_csv(CSV_PATH)
print(f"Forme du dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Forme du dataset : 55500 lignes × 15 colonnes


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


## Schéma brut

In [2]:
info_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null": df.isna().sum(),
    "n_unique": df.nunique(),
    "exemple": df.iloc[0].astype(str).values,
})
info_df

,dtype,non_null,null,n_unique,exemple
Name,str,55500,0,49992,Bobby JacksOn
Age,int64,55500,0,77,30
Gender,str,55500,0,2,Male
Blood Type,str,55500,0,8,B-
Medical Condition,str,55500,0,6,Cancer
Date of Admission,str,55500,0,1827,2024-01-31
Doctor,str,55500,0,40341,Matthew Smith
Hospital,str,55500,0,39876,Sons and Miller
Insurance Provider,str,55500,0,5,Blue Cross
Billing Amount,float64,55500,0,50000,18856.281305978155


## Doublons

In [3]:
n_dup_strict = df.duplicated().sum()
print(f"Doublons stricts (toutes colonnes) : {n_dup_strict}")

if "Name" in df.columns:
    n_dup_name = df["Name"].duplicated().sum()
    print(f"Lignes avec un Name déjà vu : {n_dup_name}")

Doublons stricts (toutes colonnes) : 534
Lignes avec un Name déjà vu : 5508


## Valeurs distinctes des champs de catégorie

In [5]:
for col in df.columns:
    if df[col].dtype == "object" and df[col].nunique() < 30:
        print(f"\n--- {col} ({df[col].nunique()} valeurs uniques) ---")
        print(df[col].value_counts(dropna=False).head(15))

## Champs date

In [6]:
date_candidates = [c for c in df.columns if "date" in c.lower() or "admission" in c.lower() or "discharge" in c.lower()]
print(f"Colonnes candidates dates : {date_candidates}")

for col in date_candidates:
    print(f"\n--- {col} ---")
    print(f"Type pandas brut : {df[col].dtype}")
    print(f"Exemples : {df[col].head(3).tolist()}")
    parsed = pd.to_datetime(df[col], errors="coerce")
    print(f"Parsing réussi : {parsed.notna().sum()} / {len(df)}")
    print(f"Min : {parsed.min()} | Max : {parsed.max()}")

Colonnes candidates dates : ['Date of Admission', 'Admission Type', 'Discharge Date']

--- Date of Admission ---
Type pandas brut : str
Exemples : ['2024-01-31', '2019-08-20', '2022-09-22']
Parsing réussi : 55500 / 55500
Min : 2019-05-08 00:00:00 | Max : 2024-05-07 00:00:00

--- Admission Type ---
Type pandas brut : str
Exemples : ['Urgent', 'Emergency', 'Emergency']
Parsing réussi : 0 / 55500
Min : NaT | Max : NaT

--- Discharge Date ---
Type pandas brut : str
Exemples : ['2024-02-02', '2019-08-26', '2022-10-07']
Parsing réussi : 55500 / 55500
Min : 2019-05-09 00:00:00 | Max : 2024-06-06 00:00:00


C:\Users\Paulo\AppData\Local\Temp\ipykernel_15896\2996053726.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")


## Numériques

In [7]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,55500,49992,DAvId muNoZ,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,55500.0,NaN,NaN,NaN,51.539459,19.602454,13.0,35.0,52.0,68.0,89.0
Gender,55500,2,Male,27774,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Blood Type,55500,8,A-,6969,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Medical Condition,55500,6,Arthritis,9308,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date of Admission,55500,1827,2024-03-16,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Doctor,55500,40341,Michael Smith,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital,55500,39876,LLC Smith,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Insurance Provider,55500,5,Cigna,11249,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Billing Amount,55500.0,NaN,NaN,NaN,25539.316097,14211.454431,-2008.49214,13241.224652,25538.069376,37820.508436,52764.276736


## Test connexion MongoDB

In [8]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)
print(client.admin.command("ping"))
print("Bases existantes :", client.list_database_names())

{'ok': 1.0}
Bases existantes : ['admin', 'config', 'healthcare_db', 'local']


## Nettoyage MongoDB

In [9]:
# Nettoyage : on repart d'une base vierge
client.drop_database("healthcare_db")
print("Bases après nettoyage :", client.list_database_names())

Bases après nettoyage : ['admin', 'config', 'local']


## Prototype de transformation sur 10 lignes

In [10]:
# 1. Échantillon de 10 lignes
sample = df.head(10).copy()

# 2. Renommage snake_case
RENAME_MAP = {
    "Name": "name",
    "Age": "age",
    "Gender": "gender",
    "Blood Type": "blood_type",
    "Medical Condition": "medical_condition",
    "Date of Admission": "date_of_admission",
    "Doctor": "doctor",
    "Hospital": "hospital",
    "Insurance Provider": "insurance_provider",
    "Billing Amount": "billing_amount",
    "Room Number": "room_number",
    "Admission Type": "admission_type",
    "Discharge Date": "discharge_date",
    "Medication": "medication",
    "Test Results": "test_results",
}
sample = sample.rename(columns=RENAME_MAP)

# 3. Normalisation casse (Title Case) sur les champs nominatifs
#    .str.title() gère bien "BoBby JaCkSoN" -> "Bobby Jackson"
sample["name"] = sample["name"].str.strip().str.title()
sample["doctor"] = sample["doctor"].str.strip().str.title()

# 4. Parsing explicite des dates (format ISO YYYY-MM-DD)
sample["date_of_admission"] = pd.to_datetime(sample["date_of_admission"], format="%Y-%m-%d")
sample["discharge_date"] = pd.to_datetime(sample["discharge_date"], format="%Y-%m-%d")

# 5. Suppression des doublons stricts (sur l'échantillon : aucun normalement)
before = len(sample)
sample = sample.drop_duplicates()
print(f"Doublons retirés sur l'échantillon : {before - len(sample)}")

# 6. Conversion en list[dict] (format attendu par insert_many)
documents = sample.to_dict(orient="records")

# 7. Inspection : on regarde le premier document
import pprint
pprint.pprint(documents[0])

Doublons retirés sur l'échantillon : 0
{'admission_type': 'Urgent',
 'age': 30,
 'billing_amount': 18856.281305978155,
 'blood_type': 'B-',
 'date_of_admission': Timestamp('2024-01-31 00:00:00'),
 'discharge_date': Timestamp('2024-02-02 00:00:00'),
 'doctor': 'Matthew Smith',
 'gender': 'Male',
 'hospital': 'Sons and Miller',
 'insurance_provider': 'Blue Cross',
 'medical_condition': 'Cancer',
 'medication': 'Paracetamol',
 'name': 'Bobby Jackson',
 'room_number': 328,
 'test_results': 'Normal'}


## Prototype d'insertion + vérification typage

In [11]:
# Base/collection dédiées au prototype, distinctes de la base finale
proto_db = client["healthcare_proto"]
proto_collection = proto_db["patients"]

# On vide pour repartir propre à chaque exécution
proto_collection.drop()

# Insertion
result = proto_collection.insert_many(documents)
print(f"Documents insérés : {len(result.inserted_ids)}")
print(f"Premier _id généré : {result.inserted_ids[0]}")

Documents insérés : 10
Premier _id généré : 69f10bca974f15732e733d92


In [12]:
# Relecture : on récupère un document et on inspecte les types Mongo
doc = proto_collection.find_one()
print("\n=== Document relu depuis MongoDB ===")
for key, value in doc.items():
    print(f"  {key:25s} | {type(value).__name__:15s} | {value!r}")


=== Document relu depuis MongoDB ===
  _id                       | ObjectId        | ObjectId('69f10bca974f15732e733d92')
  name                      | str             | 'Bobby Jackson'
  age                       | int             | 30
  gender                    | str             | 'Male'
  blood_type                | str             | 'B-'
  medical_condition         | str             | 'Cancer'
  date_of_admission         | datetime        | datetime.datetime(2024, 1, 31, 0, 0)
  doctor                    | str             | 'Matthew Smith'
  hospital                  | str             | 'Sons and Miller'
  insurance_provider        | str             | 'Blue Cross'
  billing_amount            | float           | 18856.281305978155
  room_number               | int             | 328
  admission_type            | str             | 'Urgent'
  discharge_date            | datetime        | datetime.datetime(2024, 2, 2, 0, 0)
  medication                | str             | 'Paracetamol'

In [13]:
# Comptage et stats
print(f"\nNombre de documents dans la collection : {proto_collection.count_documents({})}")

# Test de requête simple
pipeline = [
    {"$group": {
        "_id": "$medical_condition",
        "count": {"$sum": 1},
        "avg_age": {"$avg": "$age"},
    }},
    {"$sort": {"count": -1}},
]
print("\nRépartition par medical_condition (sur 10 docs) :")
for row in proto_collection.aggregate(pipeline):
    print(f"  {row}")


Nombre de documents dans la collection : 10

Répartition par medical_condition (sur 10 docs) :
  {'_id': 'Cancer', 'count': 4, 'avg_age': 37.75}
  {'_id': 'Asthma', 'count': 2, 'avg_age': 59.0}
  {'_id': 'Obesity', 'count': 2, 'avg_age': 69.0}
  {'_id': 'Diabetes', 'count': 2, 'avg_age': 24.5}


## Nettoyage du prototype

In [14]:
# Nettoyage : on supprime la base proto, on repartira propre dans le script
client.drop_database("healthcare_proto")
print("Bases après nettoyage :", client.list_database_names())

Bases après nettoyage : ['admin', 'config', 'local']


## Vérifications post-migration

In [15]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
collection = client["healthcare_db"]["patients"]

# 1. Comptage
print(f"Documents : {collection.count_documents({})}")

# 2. Indexes
print("\nIndex en place :")
for name, info in collection.index_information().items():
    print(f"  {name}: {info['key']}")

# 3. Échantillon
print("\nÉchantillon :")
import pprint
pprint.pprint(collection.find_one())

# 4. Démonstration que les index servent (explain)
print("\nPlan d'exécution d'une requête sur medical_condition :")
explain = collection.find({"medical_condition": "Cancer"}).explain()
winning_stage = explain["queryPlanner"]["winningPlan"]
print(f"  Stage gagnant : {winning_stage}")

Documents : 54966

Index en place :
  _id_: [('_id', 1)]
  idx_medical_condition: [('medical_condition', 1)]
  idx_date_of_admission_desc: [('date_of_admission', -1)]
  idx_hospital: [('hospital', 1)]
  idx_age_gender: [('age', 1), ('gender', 1)]

Échantillon :
{'_id': ObjectId('69f10e6318bb9706480b1d06'),
 'admission_type': 'Urgent',
 'age': 30,
 'billing_amount': 18856.281305978155,
 'blood_type': 'B-',
 'date_of_admission': datetime.datetime(2024, 1, 31, 0, 0),
 'discharge_date': datetime.datetime(2024, 2, 2, 0, 0),
 'doctor': 'Matthew Smith',
 'gender': 'Male',
 'hospital': 'Sons and Miller',
 'insurance_provider': 'Blue Cross',
 'medical_condition': 'Cancer',
 'medication': 'Paracetamol',
 'name': 'Bobby Jackson',
 'room_number': 328,
 'test_results': 'Normal'}

Plan d'exécution d'une requête sur medical_condition :
  Stage gagnant : {'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'medical_condition': 1}, 'indexName': 'idx_medical_condition',